# Construction de la couche Silver KBO : transformation de `entreprise` en `entreprise_silver`

## 0. Contexte : bronze vs silver

La couche **bronze** (`entreprise`) conserve les données sous forme **brute** :

- des codes non traduits (`Status="AC"`, `TypeOfAddress="REGO"`...) ;
- des tableaux indexés `0/1/2/3` sans clé porteuse de sens ;
- des champs dupliqués par langue (`CountryNL`/`CountryFR`, `MunicipalityNL`/`MunicipalityFR`...) ;
- des activités qui apparaissent en double sous plusieurs versions NACE (2003, 2008, 2025) pour une même réalité.

La couche **silver** prend en charge toutes ces transformations.

### Ce que « tout ça » signifie concrètement

Le bronze répond à *« où sont les données ? »*. Le silver répond à *« que
signifient-elles ? »*. Quatre transformations, et une règle transversale.

| # | Problème du bronze | Réponse du silver |
|---|---|---|
| 1 | `Status="AC"`, `JuridicalForm="014"` | traduction en français via `kbo_code` |
| 2 | `denominations[0]`, `denominations[1]`... | dictionnaire clé par **type traduit** |
| 3 | `MunicipalityNL` **et** `MunicipalityFR` | une seule langue, colonne unique |
| 4 | la même activité répétée en NACE 2003 / 2008 / 2025 | dédoublonnage, version la plus récente retenue |

**Règle transversale : le silver ne conserve que ce qui a du sens pour un
lecteur.** Un champ vide est omis plutôt que maintenu à vide, `EntityContact`
est supprimé, et le `NaceCode` brut est écarté dès que sa description est résolue.

> **Pourquoi ne pas tout faire en une seule couche ?** Parce que les deux étapes
> n'ont ni le même rythme ni le même risque. Le bronze est coûteux (~1 h de
> jointures) mais stable. Le silver est rapide et susceptible d'évoluer : une
> traduction à corriger, une règle métier qui change. Les séparer permet de
> rejouer le silver en quelques minutes **sans toucher au bronze**. C'est tout
> l'intérêt de l'architecture en médaillon.

---

## Configuration

In [1]:
import json
import os
import re
import time

import pymongo

MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27017")
DB_NAME = os.getenv("MONGO_DB", "kbo")
SOURCE = "entreprise"           # collection bronze (notebook precedent)
TARGET = "entreprise_silver"    # collection silver (produite ici)

client = pymongo.MongoClient(MONGO_URI, socketTimeoutMS=None)
db = client[DB_NAME]

def show(document) -> None:
    print(json.dumps(document, indent=2, ensure_ascii=False, default=str))

print("bronze :", SOURCE, f"({db[SOURCE].count_documents({}):,} documents)")
print("silver :", TARGET)

bronze : entreprise (1,955,776 documents)
silver : entreprise_silver


---

## 1. Lecture du bronze

Chargement de `entreprise` et de `kbo_code` filtré sur `Language="FR"`.

### Le référentiel tient en mémoire : on en tire parti

`kbo_code` compte 21 468 lignes, dont **7 156 en français**. C'est négligeable.
Plutôt que d'ajouter un `$lookup` sur `kbo_code` pour chaque code à traduire —
plus de 40 millions à résoudre au total — on charge la table **une fois** dans un
dictionnaire Python indexé par `(Category, Code)`.

Chaque traduction devient un accès en O(1), sans aller-retour réseau. C'est le
pattern classique du *broadcast join* : quand un côté de la jointure est petit, on
le réplique en mémoire plutôt que de le joindre à chaque requête.

Deux points d'implémentation à retenir :

- **`(Category, Code)` en clé, pas `Code` seul.** Le code `001` existe dans
  `ActivityGroup` (« Activités TVA »), dans `TypeOfDenomination` (« Dénomination »)
  et dans `JuridicalSituation` (« Situation normale »). Sans la catégorie, on
  risque de traduire un code avec la mauvaise signification.
- **`.strip()` sur la description.** Le code `TypeOfDenomination=004` vaut
  `" Dénomination de la succursale"` avec un espace initial dans la source. Sans
  nettoyage, il produirait une clé de dictionnaire incorrecte.

In [2]:
LANGUAGE = "FR"

def load_code_table(language: str = LANGUAGE) -> dict[tuple[str, str], str]:
    """`kbo_code` filtre sur une langue, indexe en memoire par (categorie, code)."""
    return {
        (doc["Category"], doc["Code"]): doc["Description"].strip()
        for doc in db.kbo_code.find(
            {"Language": language},
            {"_id": 0, "Category": 1, "Code": 1, "Description": 1},
        )
    }


CODES = load_code_table()

def translate(category: str, code, default=None):
    """Libelle francais d'un code ; `default` si le code est vide ou inconnu."""
    if not code:
        return default
    return CODES.get((category, code), default)


print(f"{len(CODES):,} codes charges en memoire\n")
for category, code in [("Status", "AC"), ("TypeOfAddress", "REGO"),
                       ("TypeOfDenomination", "001"), ("Language", "2"),
                       ("ActivityGroup", "006"), ("JuridicalForm", "014")]:
    print(f"  {category:<20} {code:<5} -> {translate(category, code)!r}")

print("\nle meme code, trois sens differents :")
for category in ("ActivityGroup", "TypeOfDenomination", "JuridicalSituation"):
    print(f"  ({category:<20}, '001') -> {translate(category, '001')!r}")

10,641 codes charges en memoire

  Status               AC    -> 'Actif'
  TypeOfAddress        REGO  -> 'Siège'
  TypeOfDenomination   001   -> 'Dénomination'
  Language             2     -> 'néerlandais'
  ActivityGroup        006   -> 'Activités ONSS'
  JuridicalForm        014   -> 'Société anonyme'

le meme code, trois sens differents :
  (ActivityGroup       , '001') -> 'Activités TVA'
  (TypeOfDenomination  , '001') -> 'Dénomination'
  (JuridicalSituation  , '001') -> 'Création juridique'


Document bronze de référence, pour comparaison avec les transformations qui suivent :

In [3]:
REFERENCE = "0200.245.711"
bronze = db[SOURCE].find_one({"_id": REFERENCE})

print("champs plats :", {k: v for k, v in bronze.items()
                         if not isinstance(v, list)})
print("\ndenominations brutes :")
show(bronze["denominations"])

champs plats : {'_id': '0200.245.711', 'EnterpriseNumber': '0200.245.711', 'Status': 'AC', 'JuridicalSituation': '012', 'TypeOfEnterprise': '2', 'JuridicalForm': '116', 'JuridicalFormCAC': '', 'StartDate': '01-01-1922'}

denominations brutes :
[
  {
    "_id": "6a69f7ecbbadb5ac70a6631e",
    "EntityNumber": "0200.245.711",
    "Language": "2",
    "TypeOfDenomination": "001",
    "Denomination": "Intercommunaal Sanatorium Denderoord"
  },
  {
    "_id": "6a69f7ecbbadb5ac70a6631f",
    "EntityNumber": "0200.245.711",
    "Language": "2",
    "TypeOfDenomination": "002",
    "Denomination": "DENDEROORD"
  }
]


---

## 2. Champs scalaires codés

`Status`, `JuridicalSituation`, `TypeOfEnterprise`, `JuridicalForm`, `JuridicalFormCAC` : cinq champs plats, chacun traduit indépendamment via `kbo_code`. Tout champ absent ou vide dans le bronze (ex. `JuridicalFormCAC=""`) doit être **omis** du silver plutôt que d'y figurer traduit en valeur nulle.

Chaque champ est décrit par un triplet **(nom de sortie, champ bronze, catégorie
`kbo_code`)**. Deux points à noter :

- `JuridicalFormCAC` se traduit avec la catégorie **`JuridicalForm`** : c'est la
  même nomenclature de formes juridiques, dans un contexte différent. Le nom du
  champ et celui de la catégorie ne coïncident donc pas toujours.
- L'omission est obtenue naturellement par une compréhension de dictionnaire avec
  walrus : si `translate` renvoie `None` (code vide ou inconnu), la clé n'est tout
  simplement jamais créée. Aucun `if` imbriqué, aucune valeur `None` résiduelle.

Les champs sont déclarés dans l'ordre alphabétique de leur nom de sortie, ce qui
garantit un ordre de clés stable et prévisible dans le document final.

In [4]:
# (nom de sortie, champ bronze, categorie kbo_code)
SCALAR_FIELDS = (
    ("juridicalForm",      "JuridicalForm",      "JuridicalForm"),
    ("juridicalFormCAC",   "JuridicalFormCAC",   "JuridicalForm"),
    ("juridicalSituation", "JuridicalSituation", "JuridicalSituation"),
    ("status",             "Status",             "Status"),
    ("typeOfEnterprise",   "TypeOfEnterprise",   "TypeOfEnterprise"),
)

def clean_scalars(bronze: dict) -> dict:
    """Traduit les 5 champs plats ; un champ vide ou inconnu est omis."""
    return {
        name: label
        for name, field, category in SCALAR_FIELDS
        if (label := translate(category, bronze.get(field)))
    }


bronze = db[SOURCE].find_one({"_id": REFERENCE})
print("bronze :", {f: bronze.get(f) for _, f, _ in SCALAR_FIELDS})
print("\nsilver :")
show(clean_scalars(bronze))
print("\n-> JuridicalFormCAC valait \"\" : la cle est absente, et non traduite en null.")

bronze : {'JuridicalForm': '116', 'JuridicalFormCAC': '', 'JuridicalSituation': '012', 'Status': 'AC', 'TypeOfEnterprise': '2'}

silver :
{
  "juridicalForm": "Société coopérative de droit public (ancien statut)",
  "juridicalSituation": "Dissolution volontaire – liquidation",
  "status": "Actif",
  "typeOfEnterprise": "Personne morale"
}

-> JuridicalFormCAC valait "" : la cle est absente, et non traduite en null.


Aperçu du document silver partiel à ce stade :

In [5]:
partial = {"_id": bronze["_id"], "enterpriseNumber": bronze["EnterpriseNumber"],
           "startDate": bronze["StartDate"], **clean_scalars(bronze)}
show(partial)

{
  "_id": "0200.245.711",
  "enterpriseNumber": "0200.245.711",
  "startDate": "01-01-1922",
  "juridicalForm": "Société coopérative de droit public (ancien statut)",
  "juridicalSituation": "Dissolution volontaire – liquidation",
  "status": "Actif",
  "typeOfEnterprise": "Personne morale"
}


---

## 3. Dénominations : tableau → dictionnaire `{type traduit: {language, denomination}}`

Chaque entrée doit être indexée par son `TypeOfDenomination` **traduit**.

### Pourquoi un dictionnaire plutôt qu'un tableau

Dans le bronze, lire l'abréviation d'une entreprise impose de **parcourir** le
tableau en testant `TypeOfDenomination == "002"`. Dans le silver, il suffit de
`doc["denominations"]["Abréviation"]`. La donnée n'a pas changé, sa
**structure d'accès** oui — et c'est tout l'intérêt d'une base documentaire.

Le type devient la clé ; le répéter dans la valeur n'aurait plus aucun sens. Il
ne reste que `{language, denomination}`, avec la langue elle aussi traduite
(`"2"` → `"néerlandais"`).

**Conflit de clés** : une entreprise peut avoir deux dénominations du même type
(deux langues, par exemple). Un dictionnaire ne conserve qu'une valeur par clé —
la spec tranche : *le dernier gagne*. C'est le comportement naturel d'une
affectation en boucle, aucun code supplémentaire n'est requis ; mais c'est une
**perte d'information assumée**, qu'il vaut mieux avoir choisie que subie.

In [6]:
def clean_denominations(rows) -> dict:
    """Tableau -> dict {type traduit: {language, denomination}} ; dernier gagne."""
    result = {}
    for row in rows:
        label = translate("TypeOfDenomination", row.get("TypeOfDenomination"))
        if not label:
            continue
        result[label] = {
            "language": translate("Language", row.get("Language"), ""),
            "denomination": row.get("Denomination", ""),
        }
    return result


print("bronze :")
show(bronze["denominations"])
print("\nsilver :")
show(clean_denominations(bronze["denominations"]))

bronze :
[
  {
    "_id": "6a69f7ecbbadb5ac70a6631e",
    "EntityNumber": "0200.245.711",
    "Language": "2",
    "TypeOfDenomination": "001",
    "Denomination": "Intercommunaal Sanatorium Denderoord"
  },
  {
    "_id": "6a69f7ecbbadb5ac70a6631f",
    "EntityNumber": "0200.245.711",
    "Language": "2",
    "TypeOfDenomination": "002",
    "Denomination": "DENDEROORD"
  }
]

silver :
{
  "Dénomination": {
    "language": "néerlandais",
    "denomination": "Intercommunaal Sanatorium Denderoord"
  },
  "Abréviation": {
    "language": "néerlandais",
    "denomination": "DENDEROORD"
  }
}


---

## 4. Adresses : tableau → dictionnaire `{type traduit: {country, street...}}`

Même principe de clé traduite que pour les dénominations, avec deux règles supplémentaires :

- **`country`** : `CountryFR` nettoyé des mentions entre parenthèses (`"France (Métropole)"` → `"France"`) et des espaces multiples ; si le résultat est vide, la valeur par défaut est `"Belgique"`.
- **Champs vides omis** : `box`, `zipcode`... laissés vides ne doivent pas figurer dans le document silver.

### Trois décisions de conception

**1. On retient le français, on écarte le néerlandais.** Le bronze porte
`MunicipalityNL` *et* `MunicipalityFR`. La couche silver est francophone (comme
`kbo_code` filtré sur `FR`) : seules les colonnes `*FR` sont conservées. C'est un
choix éditorial, pas une perte — le bronze reste disponible pour produire une
variante néerlandophone.

**2. `country` vide signifie « Belgique ».** Dans l'export KBO, le pays n'est
renseigné **que pour les adresses étrangères**. Une valeur vide n'est donc pas une
donnée manquante : c'est une information implicite rendue explicite. C'est
exactement le type de convention métier qu'une couche silver doit absorber afin que
le consommateur n'ait pas à la connaître.

**3. Les mentions entre parenthèses sont du bruit.** `"France (Métropole)"` et
`"France (Départements d'outre-mer)"` désignent le même pays pour quiconque
souhaite compter des entreprises par pays. On les supprime, puis on normalise les
espaces résiduels.

> ⚠️ **Divergence assumée entre l'énoncé et ses exemples.** La spec écrite demande
> d'omettre les champs vides (« `box`, `zipcode`... vides ne doivent pas apparaître »),
> mais **tous** les documents d'exemple de l'énoncé affichent `"box": ""`. Les deux
> sont incompatibles. C'est **la règle écrite** qui est implémentée, l'arbitrage étant
> isolé dans une constante — basculer d'un comportement à l'autre ne demande qu'une
> ligne. Les deux sorties sont présentées ci-dessous.

In [7]:
# La spec ecrite demande d'omettre les champs vides ; les exemples de l'enonce
# conservent `box: ""`. Constante = arbitrage explicite et reversible.
OMIT_EMPTY_ADDRESS_FIELDS = True

DEFAULT_COUNTRY = "Belgique"
_PARENTHESES = re.compile(r"\([^)]*\)")
_MULTI_SPACE = re.compile(r"\s+")

ADDRESS_FIELDS = (
    ("zipcode",      "Zipcode"),
    ("municipality", "MunicipalityFR"),
    ("street",       "StreetFR"),
    ("houseNumber",  "HouseNumber"),
    ("box",          "Box"),
)

def clean_country(raw: str | None) -> str:
    """'France (Metropole)' -> 'France' ; vide -> 'Belgique'."""
    country = _MULTI_SPACE.sub(" ", _PARENTHESES.sub(" ", raw or "")).strip()
    return country or DEFAULT_COUNTRY


for raw in ["France (Métropole)", "  Pays-Bas  ", "", None,
            "Royaume-Uni (Grande-Bretagne et Irlande du Nord)"]:
    print(f"  {raw!r:<52} -> {clean_country(raw)!r}")

  'France (Métropole)'                                 -> 'France'
  '  Pays-Bas  '                                       -> 'Pays-Bas'
  ''                                                   -> 'Belgique'
  None                                                 -> 'Belgique'
  'Royaume-Uni (Grande-Bretagne et Irlande du Nord)'   -> 'Royaume-Uni'


In [8]:
def clean_addresses(rows) -> dict:
    """Tableau -> dict {type traduit: {country, zipcode, municipality, ...}}."""
    result = {}
    for row in rows:
        label = translate("TypeOfAddress", row.get("TypeOfAddress"))
        if not label:
            continue
        address = {"country": clean_country(row.get("CountryFR"))}
        for name, field in ADDRESS_FIELDS:
            value = (row.get(field) or "").strip()
            if value or not OMIT_EMPTY_ADDRESS_FIELDS:
                address[name] = value
        result[label] = address
    return result


print("bronze :")
show(bronze["addresses"])
print("\nsilver (regle ecrite : champs vides omis) :")
show(clean_addresses(bronze["addresses"]))

OMIT_EMPTY_ADDRESS_FIELDS = False
print("\nsilver (variante des exemples de l'enonce : box conserve) :")
show(clean_addresses(bronze["addresses"]))
OMIT_EMPTY_ADDRESS_FIELDS = True   # on retablit la regle ecrite

bronze :
[
  {
    "_id": "6a69f823bbadb5ac70d9910d",
    "EntityNumber": "0200.245.711",
    "TypeOfAddress": "REGO",
    "CountryNL": "",
    "CountryFR": "",
    "Zipcode": "9500",
    "MunicipalityNL": "Geraardsbergen",
    "MunicipalityFR": "Geraardsbergen",
    "StreetNL": "Hoge Buizemont",
    "StreetFR": "Hoge Buizemont",
    "HouseNumber": "247",
    "Box": "",
    "ExtraAddressInfo": "",
    "DateStrikingOff": ""
  }
]

silver (regle ecrite : champs vides omis) :
{
  "Siège": {
    "country": "Belgique",
    "zipcode": "9500",
    "municipality": "Geraardsbergen",
    "street": "Hoge Buizemont",
    "houseNumber": "247"
  }
}

silver (variante des exemples de l'enonce : box conserve) :
{
  "Siège": {
    "country": "Belgique",
    "zipcode": "9500",
    "municipality": "Geraardsbergen",
    "street": "Hoge Buizemont",
    "houseNumber": "247",
    "box": ""
  }
}


Vérification de la règle « pays vide → Belgique » sur une adresse réellement
étrangère (`0257.883.408`, une association turque) :

In [9]:
foreign = db[SOURCE].find_one({"_id": "0257.883.408"})
print("CountryFR brut :", repr(foreign["addresses"][0]["CountryFR"]))
show(clean_addresses(foreign["addresses"]))

CountryFR brut : 'Turquie'
{
  "Siège": {
    "country": "Turquie",
    "zipcode": "34196",
    "municipality": "yenibosna - Istamboul",
    "street": "itkib bis ticaret komplexi b/blok coban cesme mekvil sanayi/caddesi",
    "houseNumber": "0"
  }
}


---

## 5. Contacts : tableau → dictionnaire `{email, phone, web}`

`EntityContact` (indique simplement si le contact appartient à l'entreprise, un établissement ou une succursale) ne doit **jamais** figurer dans le silver. Tout contact avec une valeur vide doit être ignoré.

### Le mapping ne peut pas provenir de `kbo_code`

Réflexe naturel : traduire `ContactType` via `kbo_code` comme tout le reste. Cela
ne fonctionne pas, pour **deux** raisons vérifiées sur les données :

1. `kbo_code` traduirait `EMAIL` en `"Adresse e-mail"` et `TEL` en
   `"Numéro de téléphone"` — or la sortie attendue exige des clés techniques
   courtes : `email`, `phone`, `web`.
2. Surtout : `contact.csv` contient **quatre** valeurs distinctes —
   `EMAIL`, `TEL`, `WEB` et **`FAX`** — alors que `kbo_code` n'en documente que
   **trois**. `FAX` n'y figure pas. Une traduction par `kbo_code` ferait purement
   et simplement disparaître les numéros de fax.

D'où un mapping explicite défini en dur, complété d'un repli `.lower()` pour tout
type pouvant apparaître dans une livraison future sans rompre le pipeline.

**`EntityContact` est ignoré** : dans le bronze, un contact est déjà rattaché à
la bonne entité par construction. Connaître sa valeur (`ENT` ou `EST`) en lisant
le document d'une entreprise n'apporte aucune information — c'est une redondance
qu'on supprime.

In [10]:
CONTACT_KEYS = {"EMAIL": "email", "TEL": "phone", "WEB": "web", "FAX": "fax"}

def clean_contacts(rows) -> dict:
    """Tableau -> dict {email?, phone?, web?, fax?} ; `EntityContact` jamais lu."""
    result = {}
    for row in rows:
        value = (row.get("Value") or "").strip()
        if not value:                       # contact vide -> ignore
            continue
        contact_type = row.get("ContactType", "")
        result[CONTACT_KEYS.get(contact_type, contact_type.lower())] = value
    return result


print("types presents dans la source :", sorted(db.kbo_contact.distinct("ContactType")))
print("types documentes dans kbo_code:",
      sorted(d["Code"] for d in db.kbo_code.find({"Category": "ContactType", "Language": "FR"})))
print("-> FAX absent du referentiel : le mapping doit etre explicite.\n")

contact_demo = db[SOURCE].find_one({"_id": "0201.543.234"})
print("bronze :")
show(contact_demo["contacts"])
print("\nsilver :")
show(clean_contacts(contact_demo["contacts"]))

types presents dans la source : ['EMAIL', 'FAX', 'TEL', 'WEB']
types documentes dans kbo_code: ['EMAIL', 'TEL', 'WEB']
-> FAX absent du referentiel : le mapping doit etre explicite.

bronze :
[
  {
    "_id": "6a69f85bbbadb5ac70059d0f",
    "EntityNumber": "0201.543.234",
    "EntityContact": "ENT",
    "ContactType": "TEL",
    "Value": "071 44 00 40"
  },
  {
    "_id": "6a69f85bbbadb5ac70059d10",
    "EntityNumber": "0201.543.234",
    "EntityContact": "ENT",
    "ContactType": "FAX",
    "Value": "071 36 04 84"
  },
  {
    "_id": "6a69f85bbbadb5ac70059d11",
    "EntityNumber": "0201.543.234",
    "EntityContact": "ENT",
    "ContactType": "EMAIL",
    "Value": "officiel.ic-tibi@tibi.be"
  }
]

silver :
{
  "phone": "071 44 00 40",
  "fax": "071 36 04 84",
  "email": "officiel.ic-tibi@tibi.be"
}


---

## 6. Activités : dédoublonnage inter-versions NACE + répartition main/secondary

C'est le point le plus délicat de toute la couche silver. Une même activité réelle
est fréquemment codée sous **plusieurs versions NACE** (2003, 2008, 2025) avec des
libellés distincts qui décrivent pourtant la même réalité. Règle : dédoublonner sur
`(activityGroup, description)` et, en cas de collision, **conserver la version NACE
la plus récente**. Le `NaceCode` brut ne doit **jamais** être conservé dans le
silver — une fois `description` résolue via `Nace{version}`, le code numérique n'a
plus aucune valeur pour un lecteur humain. Le résultat est ensuite réparti en
`{main: [...], secondary: [...]}` selon `Classification`.

### Pourquoi le code seul est insuffisant

La NACE est une nomenclature d'activités **révisée périodiquement**. Un même code
numérique ne désigne pas la même réalité d'une version à l'autre :

| Code | `Nace2008` | `Nace2025` |
|---|---|---|
| `35130` | Distribution d'électricité | **Transport** d'électricité |

Traduire un `NaceCode` sans tenir compte de son `NaceVersion` produit donc des
libellés **erronés**. D'où la résolution dans la catégorie `Nace{version}` — et
d'où le fait que le code brut, une fois la description obtenue, n'a plus aucune
utilité pour un lecteur : on le supprime.

### Périmètre exact du dédoublonnage

L'énoncé dit « dédoublonner sur `(activityGroup, description)` ». Les données
imposent une précision. Dans le résultat attendu pour FLUVIUS, la paire
`("Activités TVA", "Production d'électricité")` apparaît **deux fois** :

- dans `main`, en version **2003** ;
- dans `secondary`, en version **2008**.

Or `Nace2003 40110` et `Nace2008 35110` produisent la **même chaîne exacte**
`"Production d'électricité"` (vérifié ci-dessous). Si le dédoublonnage était
global, l'entrée 2008 aurait écrasé l'entrée 2003 et une seule survivrait.

→ **Le dédoublonnage s'applique à l'intérieur de chaque bucket `main` /
`secondary`**, pas sur l'ensemble. La clé effective est
`(classification, activityGroup, description)`.

### Pourquoi les apostrophes ne doivent surtout pas être normalisées

Toujours chez FLUVIUS, `main` contient ces deux entrées :

- `"Distribution d'électricité"` (2008) — apostrophe droite `U+0027`
- `"Distribution d’électricité"` (2025) — apostrophe courbe `U+2019`

Ce sont deux chaînes **différentes**, donc deux entrées distinctes conservées. La
comparaison s'effectue sur la chaîne brute, sans normalisation Unicode : la NACE
2025 a modifié ses conventions typographiques, et « corriger » les apostrophes
fusionnerait ces entrées et divergerait du résultat attendu.

In [11]:
# La demonstration des deux points ci-dessus, sur le referentiel reel.
print("Meme code, deux versions, deux sens :")
for version in ("2008", "2025"):
    print(f"  Nace{version} 35130 -> {translate(f'Nace{version}', '35130')!r}")

print("\nDeux codes differents, la MEME chaine exacte :")
for version, nace in (("2003", "40110"), ("2008", "35110")):
    print(f"  Nace{version} {nace} -> {translate(f'Nace{version}', nace)!r}")
print("  identiques :", translate("Nace2003", "40110") == translate("Nace2008", "35110"))

print("\nConvention typographique par version :")
for version in ("2003", "2008", "2025"):
    droite = sum(1 for (cat, _), desc in CODES.items()
                 if cat == f"Nace{version}" and "\u0027" in desc)
    courbe = sum(1 for (cat, _), desc in CODES.items()
                 if cat == f"Nace{version}" and "\u2019" in desc)
    print(f"  Nace{version} : apostrophe droite {droite:>5}   apostrophe courbe {courbe:>5}")

Meme code, deux versions, deux sens :
  Nace2008 35130 -> "Distribution d'électricité"
  Nace2025 35130 -> 'Transport d’électricité'

Deux codes differents, la MEME chaine exacte :
  Nace2003 40110 -> "Production d'électricité"
  Nace2008 35110 -> "Production d'électricité"
  identiques : True

Convention typographique par version :
  Nace2003 : apostrophe droite  1504   apostrophe courbe     0
  Nace2008 : apostrophe droite  1209   apostrophe courbe     0
  Nace2025 : apostrophe droite   705   apostrophe courbe   553


In [12]:
def clean_activities(rows) -> dict:
    """Dedoublonne les activites et les repartit en {main, secondary}.

    - la description est resolue dans la categorie `Nace{version}` : un meme
      code ne veut pas dire la meme chose d'une version a l'autre ;
    - dedoublonnage sur (activityGroup, description) *a l'interieur de chaque
      classification*, la version NACE la plus recente gagne ;
    - le `NaceCode` brut n'est jamais conserve.
    """
    buckets: dict[str, dict] = {"main": {}, "secondary": {}}

    for row in rows:
        version = row.get("NaceVersion", "")
        description = translate(f"Nace{version}", row.get("NaceCode"))
        if not description:                 # code inconnu du referentiel
            continue

        group = translate("ActivityGroup", row.get("ActivityGroup"), "")
        bucket = buckets["main" if row.get("Classification") == "MAIN" else "secondary"]
        key = (group, description)

        previous = bucket.get(key)
        if previous is None or version > previous["naceVersion"]:
            bucket[key] = {"activityGroup": group,
                           "description": description,
                           "naceVersion": version}

    return {name: list(entries.values()) for name, entries in buckets.items()}

Pour FLUVIUS (`0201.311.226`), 11 activités brutes se ramènent à 11 entrées
lisibles : le dédoublonnage ne fusionne que ce qui décrit effectivement la même réalité.

In [13]:
fluvius = db[SOURCE].find_one({"_id": "0201.311.226"})

print(f"activites brutes : {len(fluvius['activities'])}")
for row in fluvius["activities"][:6]:
    print("   ", {k: v for k, v in row.items() if k != "_id"})
print("    ...")

result = clean_activities(fluvius["activities"])
print(f"\nsilver : {len(result['main'])} main + {len(result['secondary'])} secondary")
show(result)

activites brutes : 13
    {'EntityNumber': '0201.311.226', 'ActivityGroup': '006', 'NaceVersion': '2025', 'NaceCode': '35140', 'Classification': 'MAIN'}
    {'EntityNumber': '0201.311.226', 'ActivityGroup': '001', 'NaceVersion': '2025', 'NaceCode': '35220', 'Classification': 'SECO'}
    {'EntityNumber': '0201.311.226', 'ActivityGroup': '001', 'NaceVersion': '2025', 'NaceCode': '35110', 'Classification': 'SECO'}
    {'EntityNumber': '0201.311.226', 'ActivityGroup': '001', 'NaceVersion': '2025', 'NaceCode': '35140', 'Classification': 'MAIN'}
    {'EntityNumber': '0201.311.226', 'ActivityGroup': '001', 'NaceVersion': '2025', 'NaceCode': '37000', 'Classification': 'SECO'}
    {'EntityNumber': '0201.311.226', 'ActivityGroup': '001', 'NaceVersion': '2025', 'NaceCode': '61100', 'Classification': 'SECO'}
    ...

silver : 5 main + 6 secondary
{
  "main": [
    {
      "activityGroup": "Activités ONSS",
      "description": "Distribution d’électricité",
      "naceVersion": "2025"
    },
    {


> **Note sur `Classification`.** La nomenclature distingue trois valeurs : `MAIN`,
> `SECO` (secondaire) et `ANCI` (auxiliaire). L'énoncé ne prévoit que deux buckets ;
> tout ce qui n'est pas `MAIN` est donc versé dans `secondary`, évitant ainsi de
> perdre silencieusement les activités auxiliaires.

---

## 7. Établissements : mêmes règles, en plus léger

Un établissement possède ses propres dénominations/adresses/contacts/activités, à nettoyer avec **exactement les mêmes règles** que ci-dessus. Seule différence avec le document entreprise : `EnterpriseNumber` est omis (on se trouve déjà dans le document de cette entreprise, le répéter serait une redondance pure). Résultat indexé par `EstablishmentNumber`.

« Exactement les mêmes règles » se traduit littéralement en code : on **réutilise
les quatre fonctions déjà écrites**, sans les dupliquer ni les paramétrer. C'est
le bénéfice d'avoir conçu des fonctions qui reçoivent un *tableau de lignes* en
entrée plutôt qu'un document entreprise entier.

Le numéro d'établissement migre du corps du document vers la **clé** du
dictionnaire — sur le même principe que les types de dénomination à la section 3.
Et `EnterpriseNumber` disparaît : on est déjà dans le document de cette entreprise,
le répéter pour chaque établissement constituerait une redondance inutile.

In [14]:
def clean_establishment(row: dict) -> dict:
    """Memes regles que l'entreprise, sans `EnterpriseNumber` (redondant ici)."""
    document = {}
    if row.get("StartDate"):
        document["startDate"] = row["StartDate"]
    document["denominations"] = clean_denominations(row.get("denominations", ()))
    document["addresses"] = clean_addresses(row.get("addresses", ()))
    document["contacts"] = clean_contacts(row.get("contacts", ()))
    document["activities"] = clean_activities(row.get("activities", ()))
    return document


establishments = {row["EstablishmentNumber"]: clean_establishment(row)
                  for row in fluvius["establishments"]}
show(establishments)

{
  "2.158.307.210": {
    "startDate": "01-01-1968",
    "denominations": {
      "Dénomination commerciale": {
        "language": "néerlandais",
        "denomination": "FLUVIUS o.v."
      }
    },
    "addresses": {
      "Unité d'établissement": {
        "country": "Belgique",
        "zipcode": "3500",
        "municipality": "Hasselt",
        "street": "Trichterheideweg",
        "houseNumber": "8"
      }
    },
    "contacts": {},
    "activities": {
      "main": [
        {
          "activityGroup": "Activités ONSS",
          "description": "Commerce d’électricité",
          "naceVersion": "2025"
        },
        {
          "activityGroup": "Activités ONSSAPL",
          "description": "Distribution et commerce d'électricité",
          "naceVersion": "2003"
        },
        {
          "activityGroup": "Activités ONSS",
          "description": "Commerce d'électricité",
          "naceVersion": "2008"
        }
      ],
      "secondary": []
    }
  }
}


---

## 8. Succursales : traitement allégé

Une succursale ne possède jamais de dénomination ni d'activité propres : seuls `addresses` et `contacts` sont à nettoyer. Ni `EnterpriseNumber`, ni `denominations`, ni `activities` dans la sortie. Résultat indexé par `Id`.

Le bronze produit bien `denominations: []` et `activities: []` pour les
succursales — la fonction `_detail_lookups` du premier notebook applique les quatre
jointures de manière uniforme. Le silver **supprime ces deux clés** plutôt que de
les laisser vides : une clé systématiquement vide est du bruit, et sa présence
suggérerait à tort qu'une succursale *pourrait* en avoir.

C'est précisément la division du travail entre les deux couches : le bronze reste
uniforme et mécanique, le silver applique la connaissance métier.

In [15]:
def clean_branch(row: dict) -> dict:
    """Une succursale n'a jamais de denomination ni d'activite propre."""
    document = {}
    if row.get("StartDate"):
        document["startDate"] = row["StartDate"]
    document["addresses"] = clean_addresses(row.get("addresses", ()))
    document["contacts"] = clean_contacts(row.get("contacts", ()))
    return document


itkib = db[SOURCE].find_one({"_id": "0257.883.408"})
print("bronze (cles vides conservees par le pipeline de jointure) :")
show({k: v for k, v in itkib["branches"][0].items() if k != "_id"})
print("\nsilver :")
show({row["Id"]: clean_branch(row) for row in itkib["branches"]})

bronze (cles vides conservees par le pipeline de jointure) :
{
  "Id": "9.000.006.626",
  "StartDate": "01-09-1995",
  "EnterpriseNumber": "0257.883.408",
  "denominations": [],
  "addresses": [
    {
      "_id": "6a69f858bbadb5ac7005806b",
      "EntityNumber": "9.000.006.626",
      "TypeOfAddress": "ABBR",
      "CountryNL": "",
      "CountryFR": "",
      "Zipcode": "1040",
      "MunicipalityNL": "Brussel",
      "MunicipalityFR": "Bruxelles",
      "StreetNL": "Wetstraat",
      "StreetFR": "Rue de la Loi",
      "HouseNumber": "28",
      "Box": "",
      "ExtraAddressInfo": "",
      "DateStrikingOff": ""
    }
  ],
  "contacts": [],
  "activities": []
}

silver :
{
  "9.000.006.626": {
    "startDate": "01-09-1995",
    "addresses": {
      "Succursale": {
        "country": "Belgique",
        "zipcode": "1040",
        "municipality": "Bruxelles",
        "street": "Rue de la Loi",
        "houseNumber": "28"
      }
    },
    "contacts": {}
  }
}


---

## Assemblage : le document silver complet

`to_silver` se contente d'ordonner les briques précédentes. L'ordre des clés est
délibéré — identifiants en premier, puis les blocs imbriqués du plus général au
plus spécifique, enfin les champs scalaires traduits (ordre alphabétique) en fin
de document, là où ils ne perturbent pas la lecture.

Il s'agit d'une **fonction pure** : elle reçoit un document bronze et retourne un
nouveau document, sans jamais modifier son entrée. Elle est donc testable
unitairement, sans base de données, et rejouable sans effet de bord.

In [16]:
def to_silver(bronze: dict) -> dict:
    """Document bronze -> document silver (fonction pure)."""
    document = {
        "_id": bronze["_id"],
        "enterpriseNumber": bronze.get("EnterpriseNumber") or bronze["_id"],
    }
    if bronze.get("StartDate"):
        document["startDate"] = bronze["StartDate"]

    document["denominations"] = clean_denominations(bronze.get("denominations", ()))
    document["addresses"] = clean_addresses(bronze.get("addresses", ()))
    document["contacts"] = clean_contacts(bronze.get("contacts", ()))
    document["activities"] = clean_activities(bronze.get("activities", ()))
    document["establishments"] = {
        row["EstablishmentNumber"]: clean_establishment(row)
        for row in bronze.get("establishments", ())
    }
    document["branches"] = {row["Id"]: clean_branch(row)
                            for row in bronze.get("branches", ())}

    document.update(clean_scalars(bronze))
    return document


show(to_silver(db[SOURCE].find_one({"_id": "0257.883.408"})))

{
  "_id": "0257.883.408",
  "enterpriseNumber": "0257.883.408",
  "startDate": "01-09-1995",
  "denominations": {
    "Dénomination": {
      "language": "français",
      "denomination": "ASSOCIATION TURQUE DES EXPORTATEURS DE TEXTILE ET D'HABILLEMENT D'ISTANBUL - ITKIB"
    }
  },
  "addresses": {
    "Siège": {
      "country": "Turquie",
      "zipcode": "34196",
      "municipality": "yenibosna - Istamboul",
      "street": "itkib bis ticaret komplexi b/blok coban cesme mekvil sanayi/caddesi",
      "houseNumber": "0"
    }
  },
  "contacts": {},
  "activities": {
    "main": [],
    "secondary": []
  },
  "establishments": {
    "2.076.372.003": {
      "startDate": "02-05-1996",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "ITKIB"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "1040",
          "municipality": "Bruxelles

---

## 9. Écriture dans `entreprise_silver`

Snapshot complet : suppression de `entreprise_silver` puis reconstruction intégrale.

La transformation s'effectue **en streaming** : un curseur parcourt le bronze,
`to_silver` s'applique document par document, et les résultats sont envoyés par
lots de 2 000. À aucun moment plus de quelques milliers de documents ne séjournent
en mémoire — les 1,95 million ne sont jamais chargés d'un seul bloc.

`batch_size=200` sur le curseur limite la taille des lots renvoyés par le serveur :
les documents bronze sont volumineux (jusqu'à 3 Mo), un lot par défaut de 101
documents de grande taille ferait gonfler la mémoire côté client sans aucun bénéfice.

Le `drop()` initial fait de l'opération un **snapshot complet** : le résultat ne
dépend pas de l'état antérieur de la collection, rejouer la cellule donne donc
toujours le même résultat.

In [17]:
def build_silver(batch_size: int = 2_000) -> int:
    """Snapshot complet : on vide la cible puis on la reconstruit en streaming."""
    db[TARGET].drop()
    written, buffer, started = 0, [], time.perf_counter()

    for bronze_document in db[SOURCE].find(batch_size=200):
        buffer.append(to_silver(bronze_document))
        if len(buffer) >= batch_size:
            db[TARGET].insert_many(buffer, ordered=False)
            written += len(buffer)
            buffer.clear()

    if buffer:
        db[TARGET].insert_many(buffer, ordered=False)
        written += len(buffer)

    elapsed = time.perf_counter() - started
    print(f"{TARGET} : {written:,} documents en {elapsed / 60:.1f} min "
          f"({written / elapsed:,.0f}/s)")
    return written


build_silver()

entreprise_silver : 1,955,776 documents en 4.1 min (8,022/s)


1955776

In [18]:
bronze_stats = db.command("collStats", SOURCE, scale=1024 * 1024)
silver_stats = db.command("collStats", TARGET, scale=1024 * 1024)

print(f"{'':<12}{'documents':>12}{'donnees':>12}{'doc moyen':>12}")
for label, stats in (("bronze", bronze_stats), ("silver", silver_stats)):
    print(f"{label:<12}{stats['count']:>12,}{stats['size']:>11,.0f}M"
          f"{stats['avgObjSize'] / 1024:>11,.1f}K")

gain = 1 - silver_stats["size"] / bronze_stats["size"]
print(f"\n-> le silver est {gain:.0%} plus compact que le bronze "
      f"(codes resolus, doublons NACE et champs vides supprimes)")

               documents     donnees   doc moyen
bronze         1,955,776      7,160M        3.7K
silver         1,955,776      5,389M        2.8K

-> le silver est 25% plus compact que le bronze (codes resolus, doublons NACE et champs vides supprimes)


---

## 10. Vérification

Comparaison d'un document silver produit avec ce que prédit la spec ci-dessus, sur une entreprise de référence.

Vérification sur `0201.105.843` (I.D.E.A., une intercommunale de Mons avec 24
établissements), puis traduction de **chaque ligne du schéma cible de la
section 12 en assertion automatique** exécutée sur un large échantillon. Une
inspection visuelle ne prouve rien sur 1,95 million de documents.

In [19]:
show(db[TARGET].find_one({"_id": "0201.105.843"}))

{
  "_id": "0201.105.843",
  "enterpriseNumber": "0201.105.843",
  "startDate": "02-03-1956",
  "denominations": {
    "Dénomination": {
      "language": "français",
      "denomination": "\"I.D.E.A. S.C\""
    },
    "Abréviation": {
      "language": "français",
      "denomination": "I.D.E.A."
    }
  },
  "addresses": {
    "Siège": {
      "country": "Belgique",
      "zipcode": "7000",
      "municipality": "Mons",
      "street": "Rue de Nimy",
      "houseNumber": "53"
    }
  },
  "contacts": {
    "email": "officiel.ic-idea@idea.be"
  },
  "activities": {
    "main": [
      {
        "activityGroup": "Activités ONSS",
        "description": "Administration de et contribution à l’amélioration de l’efficacité des activités économiques",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Études de marché et sondages",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités TVA",
        "d

### Vérifications automatiques du schéma cible (section 12)

In [20]:
SAMPLE_SIZE = 20_000
ADDRESS_KEYS = {"country", "zipcode", "municipality", "street", "houseNumber", "box"}
ACTIVITY_KEYS = {"activityGroup", "description", "naceVersion"}

checks: dict[str, int] = {}

def fails(name: str) -> None:
    checks[name] = checks.get(name, 0) + 1

for document in db[TARGET].aggregate([{"$sample": {"size": SAMPLE_SIZE}}]):
    if not isinstance(document["_id"], str):
        fails("_id doit etre une chaine")
    if document.get("enterpriseNumber") != document["_id"]:
        fails("enterpriseNumber == _id")

    for label, entry in document["denominations"].items():
        if label.strip() != label:
            fails("cle de denomination non nettoyee")
        if set(entry) - {"language", "denomination"}:
            fails("denomination : cle inattendue")

    for entry in document["addresses"].values():
        if not entry.get("country"):
            fails("country toujours renseigne")
        if set(entry) - ADDRESS_KEYS:
            fails("adresse : cle inattendue")
        if any(value == "" for value in entry.values()):
            fails("adresse : champ vide non omis")

    if any(value == "" for value in document["contacts"].values()):
        fails("contact vide non ignore")
    if "EntityContact" in document["contacts"]:
        fails("EntityContact present")

    for bucket in ("main", "secondary"):
        seen = set()
        for entry in document["activities"][bucket]:
            if set(entry) != ACTIVITY_KEYS:
                fails("activite : cles inattendues (naceCode ?)")
            key = (entry["activityGroup"], entry["description"])
            if key in seen:
                fails(f"doublon (activityGroup, description) dans {bucket}")
            seen.add(key)

    for number, establishment in document["establishments"].items():
        if not number.startswith("2."):
            fails("etablissement : cle != EstablishmentNumber")
        if "EnterpriseNumber" in establishment:
            fails("etablissement : EnterpriseNumber redondant")

    for number, branch in document["branches"].items():
        if not number.startswith("9."):
            fails("succursale : cle != Id")
        if set(branch) - {"startDate", "addresses", "contacts"}:
            fails("succursale : denominations/activities non supprimees")

print(f"{SAMPLE_SIZE:,} documents verifies\n")
if checks:
    for name, count in sorted(checks.items()):
        print(f"  ECHEC  {name} ({count})")
else:
    print("  Toutes les regles du schema cible sont respectees.")

20,000 documents verifies

  Toutes les regles du schema cible sont respectees.


### Comparaison avant / après transformation

In [21]:
before = db[SOURCE].find_one({"_id": "0200.245.711"})
after = db[TARGET].find_one({"_id": "0200.245.711"})

print("BRONZE — adresse brute")
show({k: v for k, v in before["addresses"][0].items() if k != "_id"})
print("\nSILVER — la meme adresse")
show(after["addresses"])

print("\nBRONZE — champs plats codes")
show({k: v for k, v in before.items() if not isinstance(v, list) and k != "_id"})
print("\nSILVER — les memes champs, traduits")
show({k: v for k, v in after.items() if isinstance(v, str) and k != "_id"})

BRONZE — adresse brute
{
  "EntityNumber": "0200.245.711",
  "TypeOfAddress": "REGO",
  "CountryNL": "",
  "CountryFR": "",
  "Zipcode": "9500",
  "MunicipalityNL": "Geraardsbergen",
  "MunicipalityFR": "Geraardsbergen",
  "StreetNL": "Hoge Buizemont",
  "StreetFR": "Hoge Buizemont",
  "HouseNumber": "247",
  "Box": "",
  "ExtraAddressInfo": "",
  "DateStrikingOff": ""
}

SILVER — la meme adresse
{
  "Siège": {
    "country": "Belgique",
    "zipcode": "9500",
    "municipality": "Geraardsbergen",
    "street": "Hoge Buizemont",
    "houseNumber": "247"
  }
}

BRONZE — champs plats codes
{
  "EnterpriseNumber": "0200.245.711",
  "Status": "AC",
  "JuridicalSituation": "012",
  "TypeOfEnterprise": "2",
  "JuridicalForm": "116",
  "JuridicalFormCAC": "",
  "StartDate": "01-01-1922"
}

SILVER — les memes champs, traduits
{
  "enterpriseNumber": "0200.245.711",
  "startDate": "01-01-1922",
  "juridicalForm": "Société coopérative de droit public (ancien statut)",
  "juridicalSituation": "Di

---

## 12. Schéma attendu de `entreprise_silver`

| Champ | Type | Origine / règle |
|---|---|---|
| `_id` | string | copié tel quel du bronze |
| `enterpriseNumber` | string | `EnterpriseNumber` (ou `_id` en secours) |
| `startDate` | string, optionnel | copié si présent |
| `status`, `juridicalSituation`, `typeOfEnterprise`, `juridicalForm`, `juridicalFormCAC` | string, optionnels | traduits FR via `code.csv`, omis si absents du bronze |
| `denominations` | dict `{type traduit: {language, denomination}}` | dernier gagne en cas de type dupliqué |
| `addresses` | dict `{type traduit: {country, zipcode, municipality, street, houseNumber, box}}` | champs vides omis, `country` nettoyé + défaut `"Belgique"` |
| `contacts` | dict `{email?, phone?, web?}` | `EntityContact` jamais lu |
| `activities` | `{main: [...], secondary: [...]}` | dédoublonné par `(activityGroup, description)`, version NACE la plus récente gagne, `naceCode` brut jamais gardé |
| `establishments` | dict `{EstablishmentNumber: {startDate?, denominations, addresses, contacts, activities}}` | mêmes règles que l'entreprise, sans `EnterpriseNumber` |
| `branches` | dict `{Id: {startDate?, addresses, contacts}}` | sans `denominations` ni `activities` (une succursale n'en a jamais) |

---

## Bilan

`entreprise_silver` est directement exploitable : aucun code à décoder, aucun
tableau à parcourir pour trouver une information, aucune activité en double.

Les trois points qui exigeaient une lecture attentive des données — et pas
seulement de l'énoncé :

1. **`FAX` existe dans `contact.csv` mais pas dans `code.csv`** → le mapping des
   types de contact doit être explicite, faute de quoi des données sont perdues.
2. **Un code NACE change de sens d'une version à l'autre** (`35130` = distribution
   en 2008, transport en 2025) → la description doit être résolue dans
   `Nace{version}`, jamais dans une table unique.
3. **Le dédoublonnage porte sur chaque bucket séparément, pas sur l'ensemble** →
   démontré par `("Activités TVA", "Production d'électricité")` présent à la fois
   en `main` (2003) et en `secondary` (2008), les deux versions produisant la même
   chaîne.

Une divergence entre l'énoncé et ses exemples a été identifiée et arbitrée
explicitement (section 4, `OMIT_EMPTY_ADDRESS_FIELDS`).

**Suite logique — la couche gold.** Le silver reste un reflet fidèle du registre,
document par document. Une couche gold agrégerait pour répondre à des questions
métier : nombre d'entreprises actives par commune et par secteur, séries
temporelles de créations, tables dénormalisées prêtes pour un outil de BI.